# **Workshop -001: Extract**

**Importación de las librerias a utilizar**

In [2]:
import yaml
import psycopg2 
from psycopg2 import sql
from sqlalchemy import create_engine, text
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

**Creamos una función donde leemos el archivo de configuración de la DB y cargamos los datos de la conexión**

In [10]:
def load_config(file_path="../../config/config.yaml"):
    with open(file_path, "r") as file:
        return yaml.safe_load(file)

**Llamamos a la funcion que carga los dados de conexión a la base de datos, y creamo la conexión**

In [11]:
config = load_config()
db_config = config["database"]

db_user = db_config["user"]
db_password = db_config["password"]
db_host = db_config["host"]
db_port = db_config["port"]
db_name = db_config["name"]

conn = psycopg2.connect(
    dbname="postgres",
    user=db_user,
    password=db_password,
    host=db_host,
    port=db_port
)
conn.autocommit = True

**Creamos la base de datos en caso de que no exista**

In [12]:
db_name = "etl_project"
try:
    with conn.cursor() as cur:
        cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(db_name)))
        print(f"Base de datos '{db_name}' creada exitosamente.")
except psycopg2.errors.DuplicateDatabase:
    print(f"La base de datos '{db_name}' ya existe.")
finally:
    conn.close()

La base de datos 'etl_project' ya existe.


**Creamos las tablas necesarias a utilizar en el proyecto**

In [14]:
engine = create_engine(f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

with engine.connect() as conn:
    # Verificar si la tabla ya existe
    result = conn.execute(text("""
        SELECT EXISTS (
            SELECT 1 
            FROM information_schema.tables 
            WHERE table_schema = 'public' 
            AND table_name = 'saldos_staging'
        );
    """))
    
    table_exists = result.scalar()

    if not table_exists:
        conn.execute(text("""
            CREATE TABLE saldos_staging (
                id BIGSERIAL PRIMARY KEY,
                documento_identidad VARCHAR(20),
                nombre VARCHAR(100),
                apellido VARCHAR(100),
                sexo CHAR(1),
                estado_civil INT,
                fecha_ingreso date,
                tipo_salario INT,
                salario NUMERIC(15, 2),
                estrato INT,
                tipovehiculo INT,
                lincred INT,
                fecsolic date,
                fecaprob date,
                fecfact date,
                fecdesc date,
                fecultcau date,
                fecultpago date,
                fecvemto date,
                plazo INT,
                vlrsolicitud NUMERIC(15, 2),
                valorob NUMERIC(15, 2),
                saldot NUMERIC(15, 2),
                cuota NUMERIC(15, 2),
                tasaint NUMERIC(3, 2),
                ciclod CHAR(1),
                periodd CHAR(1),
                clacuo CHAR(1),
                clasei CHAR(1),
                clades CHAR(1),
                periodo INT,
                saldo NUMERIC(15, 2),
                saldo_inicial NUMERIC(15, 2),
                vlr_debito NUMERIC(15, 2),
                vlr_credito NUMERIC(15, 2),
                cuopen INT,
                valor_pagado NUMERIC(15, 2),
                fecha_pago VARCHAR(20),
                mora_causado NUMERIC(15, 2),
                mora_abono NUMERIC(15, 2),
                mora_saldo NUMERIC(15, 2),
                descripcion VARCHAR(100),
                codahor CHAR(1),
                debcre CHAR(1),
                fecha_registro TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            );
        """))
        conn.commit()
        print("Tabla: saldos_staging creada exitosamente en PostgreSQL.")
    else:
        print("La tabla 'saldos_staging' ya existe en PostgreSQL.")

La tabla 'saldos_staging' ya existe en PostgreSQL.


**Leemos los datos del archivo CSV, e imprimimos 15 registros**

In [16]:
df = pd.read_csv('../../data/saldos.csv', sep=';', low_memory=False, dtype={    
    'vlr_debito': 'float64',
    'vlr_credito': 'float64',
    'cuopen': 'Int64',
    'valor_pagado': 'float64',    
    'mora_causado': 'float64',
    'mora_abono': 'float64',
    'mora_saldo': 'float64'
}, encoding='ISO-8859-1')

In [17]:
print(df.sample(n=15))

        documento_identidad            nombre             apellido sexo  \
174123           1193088613   VALERIN TATIANA           ROJAS DIAZ    F   
8188                6551553    OSCAR FERNANDO          URIBE ORTIZ    M   
165986           1118300983       LINI YISETH      ORTIZ GUTIERREZ    F   
113160             66970516   PATRICIA XIMENA        NAVIA SANCHEZ    F   
160132           1118301609     KAREN TATIANA        ARIAS ESPITIA    F   
29632              16457791         ALEXANDER       VALDEZ MORALES    M   
11300              12633358  WILFREY YERLESKY     TRAVIESO MORALES    M   
46039              16719581     VICTOR MANUEL         GARCIA LOPEZ    M   
104004             36149235     MARIA EUDOXIA  CALDERON  DE GARCIA    F   
104082             36149235     MARIA EUDOXIA  CALDERON  DE GARCIA    F   
51078              16788934            OVIDIO        LOPEZ CHILITO    M   
137537           1118287583          HARRISON  MARTINEZ SANTAMARIA    M   
13100              149794

**Copiamos los datos en una nueva variable, para guardarlos en la tabla saldos_staging**

In [18]:
df_staging = df.copy()

df_staging.sample(n=15)

,documento_identidad,nombre,apellido,sexo,estado_civil,fecha_ingreso,tipo_salario,salario,estrato,tipovehiculo,...,vlr_credito,cuopen,valor_pagado,fecha_pago,mora_causado,mora_abono,mora_saldo,descripcion,codahor,debcre
74555,31467841,MARLENE,VALENCIA LLANOS,F,1,2/12/2011,4,3927000,0,0,...,0.0,12,NaN,NaN,5234.0,0.0,5234.0,SERVICIO EXEQUIAL,3,D
153649,1118292040,LEANDRO ANDRES,ESTUPI AN GOMEZ,M,4,10/09/2024,4,3927000,3,0,...,78540.0,0,78540.0,31/10/2024,78540.0,78540.0,0.0,APORTES,1,C
45936,16703691,CARLOS JULIO,VALENCIA SANCHEZ,M,2,23/04/2012,4,4091640,3,0,...,1268.0,19,1268.0,27/02/2025,1268.0,1268.0,0.0,PAPELERIA,3,D
89597,31487232,ANGELICA,MU OZ NIETO,F,5,18/07/2022,4,1800000,1,0,...,0.0,0,NaN,NaN,NaN,NaN,NaN,REVALORIZACION,3,C
118006,94366209,JORGE NELSON,JIMENEZ PABON,M,1,20/04/2012,4,1525000,3,0,...,0.0,0,NaN,NaN,61000.0,0.0,61000.0,A/PERM,2,C
5351,6534297,LINCOHN,VERGARA GUEVARA,M,4,14/03/2023,4,1677500,3,0,...,2395.0,10,2395.0,31/05/2023,2395.0,2395.0,0.0,SERVICIO EXEQUIAL,3,D
41903,16461571,OMAR ALBERTO,CAICEDO SANCHEZ,M,1,7/03/2017,4,4301900,3,0,...,66436.0,0,66436.0,28/02/2023,66436.0,66436.0,0.0,APORTES,1,C
83292,31473613,ZANDRA PATRICIA,HERNANDEZ VARGAS,F,2,24/04/2019,4,1980000,0,0,...,0.0,3,325.0,25/09/2024,NaN,NaN,NaN,PTMO DE CONSUMO,4,D
48277,16745258,ANDER FABIO,MENA BECERRA,M,2,23/03/2010,4,3927000,3,0,...,549020.0,0,549020.0,30/06/2023,NaN,NaN,NaN,A FAVOR,3,C
96509,31485418,SILVIA MARCELA,ERAZO TAPASCO,F,4,3/03/2016,4,1677500,2,0,...,157813.0,7,183373.0,31/05/2024,183373.0,183373.0,0.0,PTMO DE CONSUMO,4,D


In [19]:
print(df_staging.head())  # Para ver los primeros registros
print(df_staging.dtypes)  # Para verificar los tipos de datos


   documento_identidad nombre           apellido sexo  estado_civil  \
0               294064  ISBEL  CHAMIZO HERNANDEZ    M             4   
1               294064  ISBEL  CHAMIZO HERNANDEZ    M             4   
2               294064  ISBEL  CHAMIZO HERNANDEZ    M             4   
3               294064  ISBEL  CHAMIZO HERNANDEZ    M             4   
4               294064  ISBEL  CHAMIZO HERNANDEZ    M             4   

  fecha_ingreso  tipo_salario  salario  estrato  tipovehiculo  ...  \
0    12/12/2003             4  2600000        0             0  ...   
1    12/12/2003             4  2600000        0             0  ...   
2    12/12/2003             4  2600000        0             0  ...   
3    12/12/2003             4  2600000        0             0  ...   
4    12/12/2003             4  2600000        0             0  ...   

   vlr_credito cuopen valor_pagado  fecha_pago mora_causado mora_abono  \
0      52000.0      0      52000.0  28/02/2023      52000.0    52000.0   
1   

In [20]:
# Lista de columnas de fecha
date_columns = [
    "fecha_ingreso", "fecsolic", "fecaprob", "fecfact", "fecdesc",
    "fecultcau", "fecultpago", "fecvemto"
]

# Intentar convertir todas las columnas de fecha a datetime.date
for col in date_columns:
    if col in df_staging.columns:
        df_staging[col] = pd.to_datetime(df_staging[col], format="%d/%m/%Y", errors="coerce").dt.date

# Verificar que ahora sean del tipo correcto
print(df_staging[date_columns].dtypes)
print(df_staging[date_columns].head())

# Insertar en la base de datos
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM saldos_staging"))
    row_count = result.scalar()  

    if row_count == 0:
        df_staging.to_sql("saldos_staging", con=engine, if_exists="append", index=False)
        print("Los datos cargados desde archivo CSV, se almacenaron correctamente en la tabla: saldos_staging.")
    else:
        print(f"La tabla 'saldos_staging' ya contiene {row_count} registros. No se insertarán nuevos datos.")

fecha_ingreso    object
fecsolic         object
fecaprob         object
fecfact          object
fecdesc          object
fecultcau        object
fecultpago       object
fecvemto         object
dtype: object
  fecha_ingreso    fecsolic    fecaprob     fecfact     fecdesc   fecultcau  \
0    2003-12-12  2023-02-07  2023-02-07  2023-02-07  2023-02-28  2025-02-28   
1    2003-12-12  2023-02-07  2023-02-07  2023-02-07  2023-02-28  2025-02-28   
2    2003-12-12  2023-02-07  2023-02-07  2023-02-07  2023-02-28  2025-02-28   
3    2003-12-12  2023-02-07  2023-02-07  2023-02-07  2023-02-28  2025-02-28   
4    2003-12-12  2023-02-07  2023-02-07  2023-02-07  2023-02-28  2025-02-28   

   fecultpago fecvemto  
0  2025-01-31      NaT  
1  2025-01-31      NaT  
2  2025-01-31      NaT  
3  2025-01-31      NaT  
4  2025-01-31      NaT  
La tabla 'saldos_staging' ya contiene 176625 registros. No se insertarán nuevos datos.
